# # 09 · My own analysis: is the pregnancy myelination signal specific to white matter?

Companion to the tutorial notebook [`09_your_own_analysis`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level3).

**Kernel:** `cajal-lipidomics` · **Reads:** `06_clustered.h5ad` · **Writes:** `results/myelination_did.csv`, `figures/fig_myelination_specificity.(pdf|png)`

The tutorial's last notebook is open-ended: pick a question, choose the tools, defend the result. This is a complete worked example of that, built to be **honest about what two sections can and can't tell us**.

The paper reports that myelin sphingolipids rise in pregnant white matter. Notebook 07 saw a hint of that. Here we ask the sharper version, and we try hard to break our own answer.

# ## The plan (written before touching the data)

| | |
|---|---|
| **Question** | Does the sphingolipid "myelination" score rise in pregnancy **more in white matter than in grey matter**? |
| **Why it matters** | A white-matter-specific rise points at myelin remodeling. A rise everywhere would look more like a whole-slide technical shift. |
| **Data** | `layers["umaia"]` (normalized lipids), white matter = Allen colour `#cccccc`, `Condition`, pixel `x`/`y`. |
| **Method** | A **difference-in-differences**: (pregnant − control in white matter) − (pregnant − control in grey matter). Any shift that hits a whole section equally cancels out. Uncertainty from a **spatial block bootstrap**. |
| **What would answer it** | Decided now: a positive difference-in-differences whose 95% interval excludes 0, holding up when we use each sphingolipid class separately, and **not** appearing for a non-myelin lipid class (PCs). |
| **What could prove me wrong** | The interval straddles 0; or PCs show the same "effect" (then it's a white-vs-grey artefact, not myelin biology); or only one lipid drives it. |
| **Limits** | One section per condition: no biological replicates, so this is a statement about **these two brains**. The bootstrap captures within-section spatial noise, not animal-to-animal variation. |

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------
from lipid_project import paths
paths.ensure_dirs()
paths.summary()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

import cajal_lipidomics as cl
from cajal_lipidomics import plotting
from cajal_lipidomics.style import FS
cl.style.set_style()

SEED = 0    # one seed for everything random (the bootstraps), so reruns give identical numbers
rng = np.random.default_rng(SEED)
print("ready. cajal_lipidomics", cl.__version__)

In [ ]:
adata = ad.read_h5ad(paths.stage("clustered"))    # the end of our pipeline (notebook 06)
U = np.asarray(adata.layers["umaia"], dtype=float)                   # the measured, normalized lipids
names = pd.Index(adata.var["lipid"].astype(str))    # lipid name per column
obs = adata.obs.copy()    # a copy, so adding columns here never touches adata

is_preg = (obs["Condition"] == "pregnant").to_numpy()    # True = pregnant pixel
is_wm = obs["allencolor"].astype(str).str.lower().eq("#cccccc").to_numpy()
print(f"{adata.n_obs:,} pixels | white matter {is_wm.mean():.1%} | pregnant {is_preg.mean():.1%}")
print("pixels per (condition, compartment):")
print(pd.crosstab(obs["Condition"], pd.Series(np.where(is_wm, "white", "grey"), index=obs.index, name="compartment")).to_string())

# ## Step 1: a score per lipid class

For a group of lipids, the score per pixel is the mean of their **z-scored** intensities. Z-scoring each lipid first (mean 0, sd 1 over all pixels of both sections) puts them on the same footing, so no single bright lipid dominates. We build one score for all sphingolipids together (the "myelination" score from notebook 07), one per sphingolipid class (for the sensitivity check), and one for **PCs** (the negative control: a big grey-matter phospholipid class, not myelin-specific).

In [ ]:
def class_score(prefixes):
    """Mean z-scored intensity, per pixel, over every lipid whose name starts with one of `prefixes`."""
    cols = [i for i, n in enumerate(names) if n.startswith(prefixes)]    # columns whose name starts with any prefix
    if not cols:    # no lipid of this class was annotated: nothing to score
        return None, []
    sub = U[:, cols]    # pixels x lipids of this class
    z = (sub - sub.mean(0)) / (sub.std(0) + 1e-12)    # z-score each lipid over all pixels
    return z.mean(1), [names[i] for i in cols]    # average across the class -> one value per pixel


SCORES = {}    # label -> per-pixel score
# (label, name prefixes) for each score we want; note "PC " with a space, so it doesn't catch PG/PE
for label, prefixes in [("all sphingolipids", ("HexCer", "Cer", "SM")), ("HexCer only", ("HexCer",)),
                        ("SM only", ("SM",)), ("Cer only", ("Cer",)), ("PC (negative control)", ("PC ",))]:
    score, used = class_score(prefixes)
    if score is not None:
        SCORES[label] = score
        print(f"{label:24s}: {len(used)} lipids -> {used[:6]}{' ...' if len(used) > 6 else ''}")
obs["myelination"] = SCORES["all sphingolipids"]    # stored for the maps

# ## Step 2: the difference-in-differences

Four means: control and pregnant, in white and in grey matter. Then

**DiD = (pregnant_white − control_white) − (pregnant_grey − control_grey)**

Why the second bracket? With one section per condition, anything that shifts a **whole slide** (leftover batch effect, slightly different section depth) shows up in grey and white matter alike. Subtracting the grey-matter change removes it. What survives is a change **specific to white matter**.

In [ ]:
def did(score, wm=is_wm, preg=is_preg):
    """Difference-in-differences of a per-pixel score: white-matter change minus grey-matter change."""
    m = lambda a, b: score[a & b].mean()    # mean score over pixels in both masks a and b
    return (m(preg, wm) - m(~preg, wm)) - (m(preg, ~wm) - m(~preg, ~wm))


s = SCORES["all sphingolipids"]    # the main score
# the four group means, laid out as a 2x2 table
four = pd.DataFrame({"control": [s[~is_preg & is_wm].mean(), s[~is_preg & ~is_wm].mean()],
                     "pregnant": [s[is_preg & is_wm].mean(), s[is_preg & ~is_wm].mean()]},
                    index=["white matter", "grey matter"])
four["change"] = four["pregnant"] - four["control"]    # pregnancy change within each compartment
print(four.round(3).to_string())
print(f"\ndifference-in-differences = {did(s):+.3f}  (in z-score units)")

# ## Step 3: honest error bars with a spatial block bootstrap

A **bootstrap** estimates uncertainty by resampling the data many times and recomputing the statistic. But pixels aren't independent: neighbours look alike (that's Moran's I from notebook 05). Resampling single pixels pretends we have far more independent evidence than we do, and gives intervals that are too narrow.

The fix: resample **spatial blocks**. We tile each section into squares of 10 x 10 pixels (250 µm), and resample whole tiles with replacement, separately within each of the four (condition, compartment) groups. Neighbouring pixels travel together, so their correlation is respected.

To keep it fast we precompute each tile's sum and pixel count per group. A resampled group mean is then just (sum of chosen tile sums) / (sum of chosen tile counts).

In [ ]:
BLOCK = 10                                                    # tile edge, in pixels (10 x 25 µm = 250 µm)
# each pixel's tile id, e.g. "75.0_12_3" = section 75, tile column 12, tile row 3
tile = (obs["SectionID"].astype(str) + "_" + (obs["x"] // BLOCK).astype(str) + "_" + (obs["y"] // BLOCK).astype(str)).to_numpy()


def tile_tables(score):
    """For each (condition, compartment) group: arrays of per-tile score sums and pixel counts."""
    out = {}
    for preg in (False, True):
        for wm in (False, True):
            m = (is_preg == preg) & (is_wm == wm)    # pixels of this (condition, compartment) group
            # per tile: sum of scores and number of pixels
            g = pd.DataFrame({"tile": tile[m], "v": score[m]}).groupby("tile")["v"].agg(["sum", "count"])
            out[(preg, wm)] = (g["sum"].to_numpy(), g["count"].to_numpy())
    return out


def bootstrap_did(score, n_boot=2000, seed=SEED):
    """Spatial block bootstrap distribution of the difference-in-differences."""
    r = np.random.default_rng(seed)    # fixed seed: the same resamples every run
    tabs = tile_tables(score)    # precompute per-tile sums and counts once
    means = {}
    for key, (sums, counts) in tabs.items():    # for each of the four groups...
        pick = r.integers(0, len(sums), size=(n_boot, len(sums)))          # resample tiles with replacement
        means[key] = sums[pick].sum(1) / counts[pick].sum(1)               # one resampled group mean per replicate
    # the DiD formula, once per bootstrap replicate (vectorised)
    return (means[(True, True)] - means[(False, True)]) - (means[(True, False)] - means[(False, False)])


boot = bootstrap_did(s)    # 2000 bootstrap DiD values
lo, hi = np.percentile(boot, [2.5, 97.5])    # the middle 95% of them = the 95% interval
n_tiles = {k: len(v[0]) for k, v in tile_tables(s).items()}    # how many tiles each group has (more tiles = steadier bootstrap)
print("tiles per group (pregnant?, white matter?):", n_tiles)
print(f"DiD = {did(s):+.3f}, 95% block-bootstrap interval [{lo:+.3f}, {hi:+.3f}]")

# For contrast: the naive pixel-level bootstrap, which ignores spatial correlation
r_ = np.random.default_rng(SEED)    # a separate generator for the naive version
naive = []
for _ in range(500):
    idx = r_.integers(0, len(s), len(s))    # resample single PIXELS with replacement (ignores spatial correlation)
    naive.append(did(s[idx], is_wm[idx], is_preg[idx]))    # recompute the DiD on the resampled pixels
print(f"pixel-level bootstrap interval (too narrow): [{np.percentile(naive, 2.5):+.3f}, {np.percentile(naive, 97.5):+.3f}]")

# The block interval is wider than the pixel-level one: that's spatial autocorrelation being taken seriously.

## Step 4: try to break it

Two checks, decided in the plan:
1. **Sensitivity:** does each sphingolipid class tell the same story on its own, or does one lipid carry everything?
2. **Negative control:** PCs aren't myelin lipids. If they show the same white-matter-specific "rise", our effect is a white-vs-grey artefact, not myelin biology.

In [ ]:
rows = []    # one row per score: its DiD and 95% interval
for label, score in SCORES.items():
    b = bootstrap_did(score)
    ci = np.percentile(b, [2.5, 97.5])    # 95% block-bootstrap interval
    # excludes_0: the whole interval sits on one side of zero
    rows.append((label, did(score), ci[0], ci[1], bool(ci[0] > 0 or ci[1] < 0)))
table = pd.DataFrame(rows, columns=["score", "DiD", "ci_low", "ci_high", "excludes_0"])
table.to_csv(paths.RESULTS / "myelination_did.csv", index=False)    # keep the numbers
print(table.round(3).to_string(index=False))

# ## Step 5: where in the white matter?

The DiD pools all tracts. Per white-matter region: the pregnancy change of the myelination score there, minus the overall grey-matter change (the same baseline correction as the DiD).

In [ ]:
grey_change = s[is_preg & ~is_wm].mean() - s[~is_preg & ~is_wm].mean()    # the grey-matter change: our baseline to subtract
per_region = []
for reg in sorted(obs.loc[is_wm, "acronym"].astype(str).unique()):    # every white-matter region
    m = (obs["acronym"].astype(str) == reg).to_numpy()    # this region's pixels
    n_c, n_p = int((m & ~is_preg).sum()), int((m & is_preg).sum())
    if n_c >= 50 and n_p >= 50:                                         # enough pixels on both sides
        # this region's change, minus the grey-matter change
        per_region.append((reg, n_c, n_p, s[m & is_preg].mean() - s[m & ~is_preg].mean() - grey_change))
per_region = pd.DataFrame(per_region, columns=["region", "n_control", "n_pregnant", "change_vs_grey"])
print(per_region.sort_values("change_vs_grey", ascending=False).round(3).to_string(index=False))

# ## Step 6: the figure

In [ ]:
fig = plt.figure(figsize=(15, 9))
gs = fig.add_gridspec(2, 3)    # a 2 x 3 grid of panels

# A, B: the myelination score on both sections, with the white matter outlined
vmin, vmax = np.quantile(s, [0.02, 0.98])    # one colour scale shared by both sections
for k, (cond, title) in enumerate([("naive", "A  control"), ("pregnant", "B  pregnant")]):
    ax = fig.add_subplot(gs[0, k])
    sub = obs[obs["Condition"] == cond]    # this section's pixels
    sc = ax.scatter(sub["x"], -sub["y"], c=sub["myelination"], cmap="magma", vmin=vmin, vmax=vmax, s=3, rasterized=True)
    # outline the white matter: give every pixel a two-way label (wm / grey) and draw where it changes
    wm_label = np.where(sub["allencolor"].astype(str).str.lower() == "#cccccc", "wm", "grey")
    plotting.allen_contours(sub.assign(acronym=wm_label), ax, color="cyan", lw=0.3, s=0.3)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"{title}: myelination score", loc="left")
cl.style.lightweight_colorbar(sc, ax, label="mean z-scored sphingolipid")

# C: the four group means
ax = fig.add_subplot(gs[0, 2])
xpos = np.arange(2)    # two bar groups: white, grey
ax.bar(xpos - 0.18, four["control"], width=0.36, color="#3366cc", label="control")
ax.bar(xpos + 0.18, four["pregnant"], width=0.36, color="#dd3333", label="pregnant")
ax.set_xticks(xpos); ax.set_xticklabels(four.index); ax.axhline(0, color="k", lw=0.6)
ax.set_ylabel("mean myelination score"); ax.legend(); ax.set_title("C  the four means", loc="left")

# D: block-bootstrap distribution of the DiD
ax = fig.add_subplot(gs[1, 0])
ax.hist(boot, bins=40, color="0.6", edgecolor="white")
ax.axvline(did(s), color="crimson", lw=2, label=f"DiD = {did(s):+.2f}")
ax.axvspan(lo, hi, color="crimson", alpha=0.1, label="95% interval")
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xlabel("difference-in-differences"); ax.legend(); ax.set_title("D  spatial block bootstrap", loc="left")

# E: DiD with intervals for each score, negative control in grey
ax = fig.add_subplot(gs[1, 1])
yy = np.arange(len(table))[::-1]    # first row of the table drawn at the top
colors = ["0.5" if "negative" in lab else "crimson" for lab in table["score"]]    # grey for the negative control, red for the sphingolipid scores
ax.errorbar(table["DiD"], yy, xerr=[table["DiD"] - table["ci_low"], table["ci_high"] - table["DiD"]],
            fmt="none", ecolor=colors, elinewidth=2)
ax.scatter(table["DiD"], yy, c=colors, zorder=3)
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(table["score"]); ax.set_xlabel("DiD (95% interval)")
ax.set_title("E  sensitivity + negative control", loc="left")

# F: per white-matter region
ax = fig.add_subplot(gs[1, 2])
pr = per_region.sort_values("change_vs_grey")    # smallest at the bottom, biggest at the top
ax.barh(pr["region"], pr["change_vs_grey"], color=np.where(pr["change_vs_grey"] > 0, "crimson", "steelblue"))
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel("pregnancy change, minus grey-matter change"); ax.set_title("F  per white-matter region", loc="left")

fig.suptitle("Is the pregnancy myelination signal specific to white matter?", fontsize=FS["l"])
plt.tight_layout()
saved = paths.savefig(fig, "fig_myelination_specificity")    # PDF (editable) + PNG in figures/
plt.show()
print("saved:", *[p.name for p in saved])

# ## Step 7: the verdict, written from the numbers

The cell below turns the pre-registered criteria from the plan into a verdict automatically, so the conclusion can't drift away from what the data actually says.

In [ ]:
main = table.set_index("score").loc["all sphingolipids"]    # the main result row
# the per-class sensitivity rows
classes = table[table["score"].isin(["HexCer only", "SM only", "Cer only"])]
neg = table[table["score"].str.contains("negative")]    # the negative-control row

# each criterion from the plan, evaluated as True/False straight from the table
checks = {
    "main DiD > 0 with a 95% interval excluding 0": bool(main["DiD"] > 0 and main["ci_low"] > 0),
    "every sphingolipid class points the same way (DiD > 0)": bool(len(classes) and (classes["DiD"] > 0).all()),
    "negative control (PC) does NOT show a positive, significant DiD":
        bool(len(neg) == 0 or not (neg["DiD"].iloc[0] > 0 and neg["ci_low"].iloc[0] > 0)),
}
for k, v in checks.items():
    print(("PASS  " if v else "FAIL  ") + k)

if all(checks.values()):
    print("\nVerdict: in these two brains, the sphingolipid (myelination) score rises in pregnancy specifically in "
          "white matter, beyond any whole-section shift, consistently across sphingolipid classes, and not for PCs.")
else:
    print("\nVerdict: the white-matter-specific myelination rise is NOT cleanly supported by these criteria. "
          "Look at which check failed: that's the interesting part.")
print("Caveat: one section per condition. This describes these two animals; replicates are needed to generalise.")

# ## Where to take it next

Natural extensions, each a small change to the code above:
- **Unsaturation:** score lipids by their double-bond count (parse names as in notebook 02) and ask whether pregnancy shifts membrane unsaturation, region by region.
- **Lipizones instead of grey/white:** run the DiD per lipizone (notebook 06 labels) against a grey-matter baseline.
- **EUCLID lipizones:** repeat using `obs["euclid_lipizone"]` and see if the conclusion depends on the clustering.
- **Annotation sensitivity:** drop lipids whose `annotation_rule` is `"ppm (tie)"` (notebook 02) and check the verdict doesn't change.

**Working with an AI assistant on extensions like these:** read every line it writes; verify against printed numbers, not its confidence; go one small cell at a time; and tell it the context (the data lives in `adata`, the uMAIA layer is the one to test on, there are exactly two sections and no replicates).